<a href="https://colab.research.google.com/github/Elsaamelia55/Elsa_Risti_Zalfa_EDA/blob/main/P05_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Membaca dataset asli kelompok
import pandas as pd
df_mentah = pd.read_csv('/diabetes_prediction_dataset.csv')
# Melihat ukuran dataset
print("Ukuran dataset:", df_mentah.shape)
# Melihat lima baris pertama
df_mentah.head()

Ukuran dataset: (100000, 9)


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


In [5]:
# Fungsi cleaning untuk dataset diabetes kelompok
import pandas as pd
def bersihkan(df_mentah):
    """Membersihkan data berdasarkan diagnosis dan mencatat setiap keputusan."""
# Salin data agar dataset asli tidak berubah
    x = df_mentah.copy()
    log = []
    def catat(langkah, jumlah, alasan):
        log.append({
            'langkah': langkah,
            'baris_terdampak': int(jumlah),
            'alasan': alasan
        })
# 1. Periksa nilai kosong teknis
    jumlah_kosong = int(x.isna().sum().sum())
    catat(
        'Pemeriksaan nilai kosong',
        jumlah_kosong,
        'Tidak ditemukan NaN; tidak dilakukan pengisian atau penghapusan nilai kosong.'
    )
# 2. Pertahankan kategori No Info
    jumlah_no_info = int(
        x['smoking_history']
        .astype(str).str.strip().str.lower()
        .eq('no info').sum()
    )
    catat(
        'Pemeriksaan No Info',
        jumlah_no_info,
        'Dipertahankan sebagai kategori tersendiri karena informasi sebenarnya tidak diketahui.'
    )
# 3. Periksa konsistensi teks tanpa mengubah kategori
    kolom_teks = x.select_dtypes(include='object').columns
    jumlah_inkonsistensi = sum(
        x[kol].nunique()
        - x[kol].astype(str).str.strip().str.lower().nunique()
        for kol in kolom_teks
    )
    catat(
        'Pemeriksaan konsistensi teks',
        jumlah_inkonsistensi,
        'Tidak ditemukan perbedaan jumlah kategori akibat spasi tepi atau kapitalisasi; nilai dipertahankan.'
    )
# 4. Periksa aturan rentang yang telah digunakan saat diagnosis
    aturan = {
        'age di luar 0–80': (x['age'] < 0) | (x['age'] > 80),
        'bmi <= 0': x['bmi'] <= 0,
        'HbA1c_level di luar 3.5–9.0':
            (x['HbA1c_level'] < 3.5) | (x['HbA1c_level'] > 9.0),
        'blood_glucose_level di luar 80–300':
            (x['blood_glucose_level'] < 80) |
            (x['blood_glucose_level'] > 300),
        'hypertension bukan 0 atau 1':
            ~x['hypertension'].isin([0, 1]),
        'heart_disease bukan 0 atau 1':
            ~x['heart_disease'].isin([0, 1]),
        'diabetes bukan 0 atau 1':
            ~x['diabetes'].isin([0, 1])
    }
    jumlah_pelanggaran = sum(int(mask.sum()) for mask in aturan.values())
    catat(
        'Pemeriksaan aturan rentang dan kode',
        jumlah_pelanggaran,
        'Tidak ditemukan pelanggaran aturan yang ditetapkan; nilai dipertahankan.'
    )
# 5. Hapus duplikat persis
    jumlah_duplikat = int(x.duplicated().sum())
    x = x.drop_duplicates()
    catat(
        'Penghapusan duplikat persis',
        jumlah_duplikat,
        'Baris identik dihapus setelah kemunculan pertama. Dataset tidak memiliki ID unik, sehingga identitas pengamatan tidak dapat dipastikan.'
    )
# Kembalikan data bersih dan catatan keputusan
    return x.reset_index(drop=True), pd.DataFrame(log)


In [6]:

# Gunakan dataset mentah yang sudah dimuat sebelumnya
df_tim = df_mentah.copy()
# Jalankan proses cleaning
bersih_tim, log_tim = bersihkan(df_tim)
# Bandingkan ukuran data
print("Ukuran data mentah :", df_tim.shape)
print("Ukuran data bersih :", bersih_tim.shape)
# Tampilkan catatan keputusan cleaning
display(log_tim)


Ukuran data mentah : (100000, 9)
Ukuran data bersih : (96146, 9)


,langkah,baris_terdampak,alasan
0,Pemeriksaan nilai kosong,0,Tidak ditemukan NaN; tidak dilakukan pengisian...
1,Pemeriksaan No Info,35816,Dipertahankan sebagai kategori tersendiri kare...
2,Pemeriksaan konsistensi teks,0,Tidak ditemukan perbedaan jumlah kategori akib...
3,Pemeriksaan aturan rentang dan kode,0,Tidak ditemukan pelanggaran aturan yang diteta...
4,Penghapusan duplikat persis,3854,Baris identik dihapus setelah kemunculan perta...


In [7]:

# Bandingkan distribusi label diabetes sebelum dan sesudah cleaning
distribusi_sebelum = (
    df_tim['diabetes']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename('sebelum_cleaning (%)')
)
distribusi_sesudah = (
    bersih_tim['diabetes']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename('sesudah_cleaning (%)')
)
perbandingan = pd.concat(
    [distribusi_sebelum, distribusi_sesudah],
    axis=1
)
display(perbandingan)

,sebelum_cleaning (%),sesudah_cleaning (%)
diabetes,,
0,91.5,91.18
1,8.5,8.82


In [8]:
# Memeriksa kondisi dataset setelah cleaning
print("=== UKURAN DATA ===")
print("Data mentah :", df_tim.shape)
print("Data bersih :", bersih_tim.shape)
print("\n=== NILAI KOSONG ===")
print("Jumlah NaN setelah cleaning:", bersih_tim.isna().sum().sum())
print("\n=== DUPLIKAT PERSIS ===")
print("Duplikat tersisa:", bersih_tim.duplicated().sum())
print("\n=== KATEGORI NO INFO ===")
jumlah_no_info_akhir = (
    bersih_tim['smoking_history']
    .astype(str)
    .str.strip()
    .str.lower()
    .eq('no info')
    .sum()
)
print("Jumlah No Info setelah cleaning:", jumlah_no_info_akhir)
print("\n=== DISTRIBUSI LABEL DIABETES ===")
display(
    bersih_tim['diabetes']
    .value_counts()
    .sort_index()
    .to_frame(name='jumlah_data')
)

=== UKURAN DATA ===
Data mentah : (100000, 9)
Data bersih : (96146, 9)

=== NILAI KOSONG ===
Jumlah NaN setelah cleaning: 0

=== DUPLIKAT PERSIS ===
Duplikat tersisa: 0

=== KATEGORI NO INFO ===
Jumlah No Info setelah cleaning: 32887

=== DISTRIBUSI LABEL DIABETES ===


,jumlah_data
diabetes,
0,87664
1,8482


In [9]:
# Menyimpan dataset hasil cleaning
bersih_tim.to_csv('diabetes_prediction_clean.csv', index=False)
# Menyimpan log keputusan cleaning
log_tim.to_csv('cleaning_log.csv', index=False)
print("Dataset bersih dan log cleaning berhasil disimpan.")

Dataset bersih dan log cleaning berhasil disimpan.


In [10]:
# Menyiapkan folder untuk hasil data bersih
import os
os.makedirs('data/processed', exist_ok=True)
# Menyimpan dataset bersih tanpa mengubah dataset mentah
bersih_tim.to_csv(
    'data/processed/diabetes_prediction_clean.csv',
    index=False
)
# Memastikan file tersimpan dan ukuran data sesuai
print("File tersimpan: data/processed/diabetes_prediction_clean.csv")
print("Ukuran dataset bersih:", bersih_tim.shape)
print("Ukuran dataset mentah:", df_mentah.shape)

File tersimpan: data/processed/diabetes_prediction_clean.csv
Ukuran dataset bersih: (96146, 9)
Ukuran dataset mentah: (100000, 9)


In [11]:
# Membandingkan statistik BMI sebelum dan sesudah pembersihan
for nama, data in [
    ('Sebelum', df_mentah),
    ('Sesudah', bersih_tim)
]:
    print(f'\n{nama} pembersihan')
    print('Mean  :', data['bmi'].mean())
    print('Median:', data['bmi'].median())
    print('Std   :', data['bmi'].std())


Sebelum pembersihan
Mean  : 27.3207671
Median: 27.32
Std   : 6.636783416649581

Sesudah pembersihan
Mean  : 27.32146111122667
Median: 27.32
Std   : 6.7677155604816965


In [12]:
from google.colab import files

files.download('data/processed/diabetes_prediction_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>